# Rerun `generate_graphs.py` with `mntss/clt-llama-3.2-1b-524k`

This notebook reruns the repository script with model `meta-llama/Llama-3.2-1B` and transcoder `mntss/clt-llama-3.2-1b-524k`. It works from the repo checkout or from a fresh `/content` runtime by cloning the repo first.

In [ ]:
!nvidia-smi

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/IamKrill1n/circuit_tracer_mod.git"
REPO_NAME = "circuit_tracer_mod"
REPO_BRANCH = "clean_up"

def find_repo_root() -> Path | None:
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    candidates.extend([Path("/home/tu/circuit_tracer_mod"), Path("/content") / REPO_NAME])
    for candidate in candidates:
        if (candidate / "generate_graphs.py").exists():
            return candidate
    return None

REPO_ROOT = find_repo_root()
clone_parent = Path("/content") if Path("/content").exists() else Path.cwd().resolve()
if REPO_ROOT is None:
    REPO_ROOT = clone_parent / REPO_NAME
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)], check=True
    )
elif REPO_ROOT == clone_parent / REPO_NAME and (REPO_ROOT / ".git").exists():
    subprocess.run(["git", "fetch", "origin", REPO_BRANCH], cwd=REPO_ROOT, check=True)
    subprocess.run(["git", "checkout", REPO_BRANCH], cwd=REPO_ROOT, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", REPO_BRANCH], cwd=REPO_ROOT, check=True)

SCRIPT = REPO_ROOT / "generate_graphs.py"
assert SCRIPT.exists(), f"missing {SCRIPT}"

print(f"repo: {REPO_ROOT}")
print(f"branch: {REPO_BRANCH}")

If this is a fresh Colab runtime, run the next cell once to install the repo dependencies. Skip it when you are already in the local `circuit` conda environment.

In [ ]:
INSTALL_DEPS = Path("/content").exists()

if INSTALL_DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT)], check=True)

`meta-llama/Llama-3.2-1B` is a gated Hugging Face model. Before graph generation, authenticate with a Hugging Face token that has Meta model access. In Colab, you can set an `HF_TOKEN` secret or paste the token when prompted.


In [ ]:
import getpass
import os
from huggingface_hub import login

hf_token = os.environ.get("HUGGINGFACE_API_KEY")
if hf_token is None:
    try:
        from google.colab import userdata

        hf_token = userdata.get("HUGGINGFACE_API_KEY")
    except Exception:
        hf_token = None

if not hf_token:
    hf_token = getpass.getpass("Hugging Face token: ")

login(token=hf_token, add_to_git_credential=False)
os.environ["HUGGINGFACE_API_KEY"] = hf_token
print("Hugging Face authentication configured.")


In [ ]:
PROMPT_FILE = REPO_ROOT / "dataset" / "analogies" / "bats_analogies.txt"
OUTPUT_DIR = REPO_ROOT / "dataset" / "analogies" / "graphs" / "clt-llama-3.2-1b-524k"

MODEL = "meta-llama/Llama-3.2-1B"
TRANSCODER = "mntss/clt-llama-3.2-1b-524k"
BACKEND = "transformerlens"
MAX_N_LOGITS = 15
DESIRED_LOGIT_PROB = 0.99
HF_REPO = None

assert PROMPT_FILE.exists(), f"missing {PROMPT_FILE}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"prompt file: {PROMPT_FILE}")
print(f"output dir: {OUTPUT_DIR}")

In [ ]:
import shutil

if shutil.which("conda"):
    python_cmd = ["conda", "run", "-n", "circuit", "python"]
else:
    python_cmd = [sys.executable]

cmd = [
    *python_cmd,
    str(SCRIPT),
    "--prompt-file",
    str(PROMPT_FILE),
    "--output-dir",
    str(OUTPUT_DIR),
    "--model",
    MODEL,
    "--transcoder",
    TRANSCODER,
    "--backend",
    BACKEND,
    "--max-n-logits",
    str(MAX_N_LOGITS),
    "--desired-logit-prob",
    str(DESIRED_LOGIT_PROB),
]

if HF_REPO:
    cmd.extend(["--hf-repo", HF_REPO])

print(" ".join(cmd))

In [ ]:
result = subprocess.run(cmd, cwd=REPO_ROOT, text=True, capture_output=True)

if result.stdout:
    print(result.stdout)

if result.stderr:
    print(result.stderr, file=sys.stderr)

result.check_returncode()

In [ ]:
import zipfile

zip_path = OUTPUT_DIR.parent / f"{OUTPUT_DIR.name}.zip"
pt_files = sorted(OUTPUT_DIR.glob("*.pt"))
assert pt_files, f"no graph files found in {OUTPUT_DIR}"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in pt_files:
        archive.write(path, arcname=f"{OUTPUT_DIR.name}/{path.name}")

print(f"zipped {len(pt_files)} graphs → {zip_path}")

try:
    from google.colab import files

    files.download(str(zip_path))
except Exception:
    print(f"Download or copy this archive from the Colab filesystem: {zip_path}")


In [ ]:
import os
import getpass
from huggingface_hub import HfApi, login

HF_DATASET_REPO = "anhtu77/analogies_BATS_circuit"
HF_PATH_IN_REPO = "analogies/graphs/clt-llama-3.2-1b-524k"

assert OUTPUT_DIR.exists(), OUTPUT_DIR
pt_files = sorted(OUTPUT_DIR.glob("*.pt"))
assert pt_files, f"no graph files found in {OUTPUT_DIR}"

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None

if not hf_token:
    hf_token = getpass.getpass("Hugging Face token: ")

login(token=hf_token, add_to_git_credential=False)

api = HfApi(token=hf_token)
api.create_repo(
    repo_id=HF_DATASET_REPO,
    repo_type="dataset",
    exist_ok=True,
    private=False,
)

api.upload_folder(
    folder_path=str(OUTPUT_DIR),
    repo_id=HF_DATASET_REPO,
    repo_type="dataset",
    path_in_repo=HF_PATH_IN_REPO,
    commit_message="upload Llama 3.2 1B 524k CLT analogy graphs",
)

print(
    f"Uploaded {OUTPUT_DIR} to "
    f"https://huggingface.co/datasets/{HF_DATASET_REPO}/tree/main/{HF_PATH_IN_REPO}"
)